In [1]:
from saf_ida import *

In [2]:
job_name = 'wall_ida'
job_config = './tests/LA/saf-ida_configuration_LA_user.json'
with open(job_config) as f:
    job_info = json.load(f)
# directory
dir_info = job_info['Directory']
work_dir = dir_info['Work']
input_dir = dir_info['Input']
output_dir = dir_info.get('Output',None)
if output_dir is None:
    output_dir = os.path.join(os.path.dirname(os.path.abspath(__file__),'Output'))
try:
    os.mkdir(f"{output_dir}")
except:
    print('runSAF_IDA: output directory already exists.')

# job type
job_type = job_info.get('Type', None)
if job_type is None:
    err_msg = 'run_saf_ida: Please specity "Type" in the configuraiton file.'
    saf_ida_job.logfile.write_msg(msg=err_msg, msg_type='ERROR')

runSAF_IDA: output directory already exists.


In [3]:
saf_ida_job = SAF_IDA(dir_info=dir_info, job_name=job_name)

2025-03-03 05:19:03.121145  RUNNING-MSG --NEW LOG STARTING FROM THIS LINE-- 


C:\Programs\saf-ida\general.py:76: FutureWarning: Starting with pandas version 3.0 all arguments of to_hdf except for the argument 'path_or_buf' will be keyword-only.
  df.to_hdf(store, 'basic', mode='a')


In [4]:
if 'SiteSpecificHazard' in job_type:
    # KZ: 03/02/25 - extending this to user-defined target
    tgt_type = job_info.get('Prediction',dict()).get('TargetType','SiteSpecific')
    if tgt_type == 'SiteSpecific':
        # get the site config.
        site_config = job_info.get('SiteSpecificHazard', None)
        if site_config is None:
            err_msg = 'run_saf_ida: SiteSpecificHazard not found in job configuration.'
            saf_ida_job.logfile.write_msg(msg=err_msg, msg_type='ERROR')
        # create site specific hazard information data
        saf_ida_job.get_site_specific_hazard(site_config=site_config)
    elif tgt_type == 'UserDefined':
        tgt_config = job_info.get('UserDefinedHazard',None)
        if tgt_config is None:
            err_msg = 'run_saf_ida: UserDefinedHazard not found in job configuration.'
            saf_ida_job.logfile.write_msg(msg=err_msg, msg_type='ERROR')
        # create user-defined hazard information data
        saf_ida_job.get_user_defined_hazard(tgt_config=tgt_config)

2025-03-03 05:19:03.811902  RUNNING-MSG SAF_IDA.get_user_defined_hazard: user-defined hazard target configured. 


In [5]:
saf_ida_job.site_data_dict['LosAngeles'].keys()

dict_keys(['Coord.', 'Number of intensity levels', 'Target type', 'Intensity Measures', 'T1 (s)', 'Spectral period (s)', 'Return period (yr)', 'Sa(T1) (g)', 'PSA (g)', 'Ds575 (s)', 'Ds595 (s)', 'Covariance'])

In [6]:
saf_ida_job.site_data_dict['LosAngeles']['Sa(T1) (g)']

[0.07345199912556123, 0.11830574324030248]

In [7]:
saf_ida_job.site_data_dict['LosAngeles']['Ds575 (s)']

[9.681863783973828, 8.999587511888892]

In [10]:
len(saf_ida_job.site_data_dict['LosAngeles']['PSA (g)'][0])

1000

In [11]:
saf_ida_job.site_data_dict['LosAngeles']['Covariance'][0][0]

[0.5771648545838386,
 0.5861902275025489,
 0.5874947153812462,
 0.5954672910116292,
 0.6065055197469028,
 0.6152643927512041,
 0.6281696113005628,
 0.647594680293629,
 0.6651840459005648,
 0.6843085617838005,
 0.6992350339071884,
 0.7172811762875175,
 0.7300635436025175,
 0.7153374476949759,
 0.6999476377939388,
 0.691579378165424,
 0.6896882738664404,
 0.6698085815984248,
 0.6589391432044219,
 0.6597234253376695,
 0.6566762608661489,
 0.6670096129037655,
 0.6836115246982558,
 0.67427738055046,
 0.6483888203198795,
 0.6351185194824126,
 0.6394015765287766,
 0.6391611195871052,
 0.6313732279872428,
 0.6187363573327186,
 0.6118522169889276,
 0.610947486687064,
 0.6229139142249811,
 0.6348241499329266,
 0.6392418602091868,
 0.6356455369533522,
 0.6204351819378002,
 0.6114068284088215,
 0.5988699032358991,
 0.5868459825210505,
 0.5818614443751348,
 0.5871430061689346,
 0.5868168198095963,
 0.579182865643191,
 0.5656385742378724,
 0.5570580620614766,
 0.552029595146222,
 0.5521212258305107,

In [5]:
#with open('./site_data.json','w') as f:
#    json.dump(saf_ida_job.site_data_dict,f,indent=2)

In [6]:
# get training config
train_config = job_info.get('Training', None)
if train_config is None:
    err_msg = 'run_saf_ida: Training not found in job configuration.'
    saf_ida_job.logfile.write_msg(msg=err_msg, msg_type='ERROR')
# create a training run
saf_ida_job.model_training(input_dir=input_dir, train_config=train_config)

Loading structural and ground motion data.
Computing SaRatio.
SaRatio computed.
Computing SaRatio.
SaRatio computed.
Computing SaRatio.
SaRatio computed.
Data loaded.
Processing collapse data.
Collapse data processed.
Median collapse Sa (g) = [2.99328307]
Computing Sa (g) for different EDPs.
Sa (g) computed.
Computing collapse model.
Collapse model computed.
Computing EDP models.
EDP models computed.


0

In [7]:
print(saf_ida_job.saf_model.col_model.model.coef_)

[1.5947922  0.31247693]


In [8]:
print(saf_ida_job.saf_model.col_model.model.intercept_)

-0.8018247435694876


In [9]:
print(np.squeeze(saf_ida_job.saf_model.saratio_pool_col))

[1.91057225 2.01262227 1.84344195 2.87541664 0.94143451 1.51467489
 1.3945518  1.28266555 1.90702325 2.19352476 1.65839751 1.4882432
 1.53803164 1.23601333 1.70062159 2.05559707 2.54612236 1.70897474
 1.3912697  1.27910832 1.3423217  1.88315156 2.02224699 1.52300887
 1.57369532 1.17708274 1.17588677 1.29018373 1.04508446 2.6469806
 1.48443454 1.19345281 2.20160514 1.2664289  1.51211985 0.96373046
 2.28394549 0.97544004 1.32805337 1.31143615 1.96457643 1.22309409
 2.06462057 1.52346589]


In [10]:
print(saf_ida_job.saf_model.gmdata.get('Ds575'))

[25.235, 27.535, 31.095000000000002, 33.64, 28.43, 30.240000000000002, 53.800000000000004, 62.7, 53.24, 67.51, 66.54, 66.6, 63.550000000000004, 44.64, 38.9, 34.45, 27.79, 28.7, 53.15, 54.43, 48.120000000000005, 55.46, 70.36, 43.71, 45.7, 45.92, 37.67, 57.050000000000004, 40.46, 42.03, 44.84, 45.97, 68.8, 70.15, 71.21000000000001, 78.76, 71.75, 35.548, 30.765, 25.71, 27.675, 33.335, 38.205, 38.85]


In [11]:
np.exp(np.mean(np.log(np.squeeze(saf_ida_job.saf_model.saratio_pool_col))))

1.5641006663681571

In [12]:
print(np.squeeze(saf_ida_job.saf_model.imcol))

[6.11 6.75 6.52 6.17 1.05 4.74 2.01 1.86 3.75 7.95 2.62 2.51 7.08 1.54
 3.59 7.03 2.91 2.75 4.73 1.8  3.48 3.68 7.12 1.79 4.35 2.02 2.28 5.36
 1.7  5.12 3.14 4.52 4.7  1.59 2.35 1.33 7.21 1.04 1.32 1.31 2.77 1.08
 2.62 1.53]


In [13]:
# get training config
pred_config = job_info.get('Prediction', None)
if pred_config is None:
    err_msg = 'run_saf_ida: Prediction not found in job configuration.'
    saf_ida_job.logfile.write_msg(msg=err_msg, msg_type='ERROR')
# create a training run
saf_ida_job.model_prediction(pred_config=pred_config, output_dir=output_dir)

SiteInfo: site data loaded.
Collapse model received.
EDP models received.
Computing GCIM targets.
GCIM targets computed.
Adjusting EDP: SDRmax-story1 for LosAngeles
Adjusted median of SDRmax-story1 at RP224: 0.0020977296903482573
Adjusted std of SDRmax-story1 at RP224: 0.22798813424815312
Adjusted median of SDRmax-story1 at RP495: 0.002746702857518543
Adjusted std of SDRmax-story1 at RP495: 0.2698943200326814
Adjusted median of SDRmax-story1 at RP975: 0.0034552664793335353
Adjusted std of SDRmax-story1 at RP975: 0.3594934957630085


C:\Users\kuans\AppData\Local\Programs\Python\Python311\Lib\site-packages\scipy\optimize\_optimize.py:941: RuntimeWarning: invalid value encountered in subtract
  np.max(np.abs(fsim[0] - fsim[1:])) <= fatol):


Adjusted median of SDRmax-story1 at RP2475: 0.005189435170180101
Adjusted std of SDRmax-story1 at RP2475: 0.6126699358348693
Adjusted median of SDRmax-story1 at RP4950: 0.007063226109000523
Adjusted std of SDRmax-story1 at RP4950: 0.7154439637676617
Adjusted median of SDRmax-story1 at RP9900: 0.00948938708779505
Adjusted std of SDRmax-story1 at RP9900: 0.7573953248926357
Adjusting EDP: SDRmax-story2 for LosAngeles
Adjusted median of SDRmax-story2 at RP224: 0.0041407677825268
Adjusted std of SDRmax-story2 at RP224: 0.2654915961147212
Adjusted median of SDRmax-story2 at RP495: 0.005270263586353585
Adjusted std of SDRmax-story2 at RP495: 0.3204214710352874
Adjusted median of SDRmax-story2 at RP975: 0.006449688221127341
Adjusted std of SDRmax-story2 at RP975: 0.365707445338016
Adjusted median of SDRmax-story2 at RP2475: 0.008583144128951656
Adjusted std of SDRmax-story2 at RP2475: 0.45933944308897584
Adjusted median of SDRmax-story2 at RP4950: 0.010435527637945264
Adjusted std of SDRmax-st